In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load
import os
import sys
import json
import random

import numpy as np 
import pandas as pd 

import librosa as lb
import librosa.feature as lf
import librosa.display as ld
import soundfile as sf
import kagglehub 


import matplotlib.pyplot as plt
from IPython.display import Audio
from tqdm import tqdm


# Kaggle Set-up
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
#os.environ['HF_TOKEN'] = user_secrets.get_secret("hf_access")
os.environ["KAGGLE_USERNAME"] = user_secrets.get_secret("kgg_user")
os.environ["KAGGLE_KEY"] = user_secrets.get_secret("kgg_key")

GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop','jazz', 'metal', 'pop', 'reggae', 'rock'] 
SR = 22050
DURATION = 30



In [3]:
def load_and_fix(path,sr=SR,duration=DURATION):
    LENGTH = sr*duration    
    y,sr = lb.load(path)
    
    # Trim or pad
    if y.shape[0] >= LENGTH:
        return y[:LENGTH]
    else:
        padding = LENGTH - y.shape[0]
        return np.pad(y,(0,padding))

try:
    dir_path = f"/kaggle/working/test-mel"
    os.makedirs(dir_path, exist_ok=True)
    print(f"Directory created at: {dir_path}")
except Exception as e:
    print(f"Error creating directory: {e}")


root_to_test = "/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup"
test_csv = pd.read_csv("/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/test.csv")
for song_id,song_path in tqdm(zip(test_csv['id'],test_csv['filename']),desc="Test mels extraction...",total=test_csv.shape[0]):
    path = os.path.join(root_to_test,song_path)
    
    y = load_and_fix(path)
    mel = lf.melspectrogram(y=y,sr=SR,n_fft=1024,hop_length=1024)
    log_mel_spec = lb.power_to_db(mel, ref=np.max)
    np.save(f"{dir_path}/mel_{song_id}.npy", log_mel_spec)


Directory created at: /kaggle/working/test-mel


Test mels extraction...: 100%|██████████| 3020/3020 [03:24<00:00, 14.79it/s]


In [4]:
# Storing to kaggle hub
handle = f'akashkumbhakar/test-mel-3020'
local_dataset= f'/kaggle/working/test-mel'

# Create a new dataset
kagglehub.dataset_upload(handle, local_dataset)

Uploading Dataset https://api.kaggle.com/datasets/akashkumbhakar/test-mel-3020 ...
More than 50 files detected, creating a zip archive...
Starting upload for file /tmp/tmpokcyopoz/archive.zip


Uploading: 100%|██████████| 1.00G/1.00G [00:13<00:00, 72.8MB/s]

Upload successful: /tmp/tmpokcyopoz/archive.zip (953MB)


Your dataset has been created.
Files are being processed...
See at: https://api.kaggle.com/datasets/akashkumbhakar/test-mel-3020


In [ ]:
import shutil
if os.path.exists(local_dataset):
    shutil.rmtree(local_dataset)
    print(f"{local_dataset} Removed")
else:
    print(f"PATH : {local_dataset} not exist OR alreaady removed.")